In [1]:
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ.get("PINECONE_API_KEY") is not None

True

In [3]:
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

In [ ]:
pc.create_index(
    name='test', dimension=1536,
    spec=ServerlessSpec(cloud='aws', region='us-east-1', concurrency=10)
)

In [5]:
index = pc.Index("my-first-index")

# Print the index statistics
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=256, total_vector_count=0, metric='cosine', namespaces=0)


In [4]:
pc.list_indexes()

IndexList([<name='pinecone-datacamp', ready=True>, <name='semantic-search-datacamp', ready=True>])

Batching Upserts

In [ ]:
import itertools


def chunks(iterable, batch_size=100): 
    it = iter(iterable) 
    chunk = tuple(itertools.islice(it, batch_size)) 
    while chunk: 
        yield chunk 
        chunk = tuple(itertools.islice(it, batch_size)) 


with pc.Index('datacamp-index', pool_thread=30):
    async_results = [index.upsert(vectors=chunk, async_req=True) for chunk in chunks(vectors, batch_size=1000)]
    [async_result.get() for async_result in async_results]

Semantic Search with Pinecone and OpenAI 

In [5]:
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [6]:
import numpy as np
from uuid import uuid4
import pandas as pd

In [7]:
pc.create_index(
    name='semantic-search-datacamp',
    dimension=1536,
    spec=ServerlessSpec(cloud='aws', region='us-east-1')
)
index = pc.Index('semantic-search-datacamp')

In [20]:
index = pc.Index('semantic-search-datacamp')
print(index.describe_index_stats())


DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=1)


In [8]:
df = pd.read_csv("netflix_titles_1000.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [17]:
# Replace missing values in text and metadata columns
columns_to_clean = [
    'show_id', 'type', 'title', 'director', 'cast',
    'country', 'date_added', 'release_year', 'rating',
    'duration', 'listed_in', 'description'
]

for col in columns_to_clean:
    df[col] = df[col].fillna('')

# Verify that description has no missing values
print("Missing description values:", df['description'].isna().sum())

Missing description values: 0


In [21]:

from uuid import uuid4

# Initialize the Pinecone index
index = pc.Index('semantic-search-datacamp')

batch_limit = 100

for start in range(0, len(df), batch_limit):
    batch = df.iloc[start:start + batch_limit]

    # Extract metadata from each row
    metadatas = [{
        "id": str(row['show_id']),
        "type": row['type'],
        "title": row['title'],
        "director": row['director'],
        "cast": row['cast'],
        "country": row['country'],
        "date_added": row['date_added'],
        "release_year": int(row['release_year']) if row['release_year'] != '' else 0,
        "rating": row['rating'],
        "duration": row['duration'],
        "listed_in": row['listed_in'],
        "description": row['description']
    } for _, row in batch.iterrows()]

    descriptions = batch['description'].tolist()

    ids = [str(uuid4()) for _ in range(len(descriptions))]

    # Generate embeddings
    response = client.embeddings.create(
        input=descriptions,
        model="text-embedding-3-small"
    )

    embeds = [x.embedding for x in response.data]

    # Upsert vectors into Pinecone
    vectors = list(zip(ids, embeds, metadatas))

    index.upsert(
        vectors=vectors,
        namespace='netflix_dataset'
    )

    print(f"Uploaded {len(vectors)} records")

Uploaded 100 records
Uploaded 100 records
Uploaded 100 records
Uploaded 100 records
Uploaded 100 records
Uploaded 100 records
Uploaded 100 records
Uploaded 100 records
Uploaded 100 records
Uploaded 100 records


In [ ]:
# Initialize the Pinecone client
index = pc.Index('semantic-search-datacamp')

query = "which movie did Adam Salky direct?"

# Create the query vector
query_response = client.embeddings.create(
    input=query,
    model="text-embedding-3-small"
)
query_emb = query_response.data[0].embedding

# Query the index and retrieve the top five most similar vectors
retrieved_docs = index.query(
    vector=query_emb,
    top_k=5,
    namespace='netflix_dataset',
    include_metadata=True
)

for result in retrieved_docs['matches']:
    print(f"{result['id']}: {round(result['score'], 2)}")
    print(f"Title: {result['metadata']['title']}")
    print('\n')

4763531a-7150-492c-ac58-418a123b8bd1: 0.43
Title: The Lincoln Lawyer


e6288626-6c60-4fd1-adb4-78a57262d4df: 0.42
Title: Dark Skies


0b93eef9-c195-4042-8c16-dab023db1080: 0.42
Title: Naruto the Movie: Ninja Clash in the Land of Snow


6cad20db-7fcc-4ba1-ac92-ad277c4bc870: 0.41
Title: Hostage House


b0b9e9d1-8e72-4316-b0e9-d38513a3bf24: 0.41
Title: Colony




In [26]:

import pandas as pd
from uuid import uuid4

text_columns = [
    'title', 'type', 'director', 'cast',
    'country', 'listed_in', 'description'
]

# Create searchable text from multiple fields
for col in text_columns:
    df[col] = df[col].fillna('').astype(str)

df['text'] = (
    "Title: " + df['title'] +
    ". Type: " + df['type'] +
    ". Director: " + df['director'] +
    ". Cast: " + df['cast'] +
    ". Country: " + df['country'] +
    ". Genres: " + df['listed_in'] +
    ". Description: " + df['description']
)

print(df[['show_id', 'text']].head())
print("Total records:", len(df))

  show_id                                               text
0      s1  Title: Dick Johnson Is Dead. Type: Movie. Dire...
1      s2  Title: Blood & Water. Type: TV Show. Director:...
2      s3  Title: Ganglands. Type: TV Show. Director: Jul...
3      s4  Title: Jailbirds New Orleans. Type: TV Show. D...
4      s5  Title: Kota Factory. Type: TV Show. Director: ...
Total records: 1000


In [27]:

index = pc.Index('semantic-search-datacamp')

index.delete(
    delete_all=True,
    namespace='netflix_dataset'
)

print("Old vectors deleted.")

Old vectors deleted.


In [29]:

batch_limit = 100

metadata_columns = [
    'type', 'title', 'director', 'cast', 'country',
    'date_added', 'release_year', 'rating',
    'duration', 'listed_in', 'description'
]

for start in range(0, len(df), batch_limit):
    batch = df.iloc[start:start + batch_limit]

    # Use Netflix show IDs as stable Pinecone vector IDs
    ids = batch['show_id'].astype(str).tolist()

    # Prepare text for embedding
    texts = batch['text'].tolist()

    # Prepare metadata, replacing missing values
    metadatas = []

    for _, row in batch.iterrows():
        metadata = {'show_id': str(row['show_id'])}

        for col in metadata_columns:
            value = row[col]

            if pd.isna(value):
                value = ''
            elif hasattr(value, 'item'):
                value = value.item()

            # Convert unsupported metadata types to strings
            if not isinstance(value, (str, int, float, bool)):
                value = str(value)

            metadata[col] = value

        metadatas.append(metadata)

    # Generate embeddings matching the 256-dimensional index
    response = client.embeddings.create(
        input=texts,
        model='text-embedding-3-small',
        dimensions=1536
    )

    embeds = [item.embedding for item in response.data]

    # Upload the vectors
    vectors = list(zip(ids, embeds, metadatas))

    index.upsert(
        vectors=vectors,
        namespace='netflix_dataset'
    )

    print(f"Uploaded {start + len(batch)} / {len(df)} records")

print("All embeddings uploaded successfully!")

Uploaded 100 / 1000 records
Uploaded 200 / 1000 records
Uploaded 300 / 1000 records
Uploaded 400 / 1000 records
Uploaded 500 / 1000 records
Uploaded 600 / 1000 records
Uploaded 700 / 1000 records
Uploaded 800 / 1000 records
Uploaded 900 / 1000 records
Uploaded 1000 / 1000 records
All embeddings uploaded successfully!


In [30]:

stats = index.describe_index_stats()
print(stats)

DescribeIndexStatsResponse(dimension=1536, total_vector_count=1000, metric='cosine', namespaces=1)


In [8]:
# Initialize the Pinecone client
index = pc.Index('semantic-search-datacamp')

query = "which movie did Adam Salky direct"

# Create the query vector
query_response = client.embeddings.create(
    input=query,
    model="text-embedding-3-small"
)
query_emb = query_response.data[0].embedding

# Query the index and retrieve the top five most similar vectors
retrieved_docs = index.query(
    vector=query_emb,
    top_k=5,
    namespace='netflix_dataset',
    include_metadata=True
)

for result in retrieved_docs['matches']:
    metadata = result['metadata']

    print(f"ID: {result['id']}")
    print(f"Score: {round(result['score'], 2)}")
    print(f"Title: {metadata.get('title', '')}")
    print(f"Director: {metadata.get('director', '')}")
    print()

ID: s28
Score: 0.39
Title: Grown Ups
Director: Dennis Dugan

ID: s294
Score: 0.37
Title: Slay
Director: Adze Ugah

ID: s8
Score: 0.35
Title: Sankofa
Director: Haile Gerima

ID: s19
Score: 0.35
Title: Intrusion
Director: Adam Salky

ID: s767
Score: 0.33
Title: Alan Saldaña: Locked Up
Director: Alex Díaz



RAG CHATBOT WITH PINECONE AND OPENAI API